#hackathon2 start


step 1: EDA Analaysis

In [1]:
import pandas as pd
import numpy as np

In [2]:
train=pd.read_csv(r'C:\Users\sunkara sri yashasvi\Downloads\Train_dataset.csv')
test=pd.read_csv(r'C:\Users\sunkara sri yashasvi\Downloads\Test_dataset.csv')

In [3]:
#shape of the data set and its info
print("train shape:",train.shape)
print("test shape:",test.shape)
print("\n")
print("info on traindataset")
train.info()
print("\n")
print("info on testdataset")
test.info()


train shape: (1966, 9)
test shape: (312, 8)


info on traindataset
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1966 entries, 0 to 1965
Data columns (total 9 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   SEQN       1954 non-null   float64
 1   RIAGENDR   1948 non-null   float64
 2   PAQ605     1953 non-null   float64
 3   BMXBMI     1948 non-null   float64
 4   LBXGLU     1953 non-null   float64
 5   DIQ010     1948 non-null   float64
 6   LBXGLT     1955 non-null   float64
 7   LBXIN      1957 non-null   float64
 8   age_group  1952 non-null   object 
dtypes: float64(8), object(1)
memory usage: 138.4+ KB


info on testdataset
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 312 entries, 0 to 311
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   SEQN      310 non-null    float64
 1   RIAGENDR  310 non-null    float64
 2   PAQ605    311 non-null    float64
 3   BMXBMI 

In [4]:
train.head(10)

,SEQN,RIAGENDR,PAQ605,BMXBMI,LBXGLU,DIQ010,LBXGLT,LBXIN,age_group
0,73564.0,2.0,2.0,35.7,110.0,2.0,150.0,14.91,Adult
1,73568.0,2.0,2.0,20.3,89.0,2.0,80.0,3.85,Adult
2,73576.0,1.0,2.0,23.2,89.0,2.0,68.0,6.14,Adult
3,73577.0,1.0,2.0,28.9,104.0,NaN,84.0,16.15,Adult
4,73580.0,2.0,1.0,35.9,103.0,2.0,81.0,10.92,Adult
5,73581.0,1.0,2.0,23.6,110.0,2.0,100.0,6.08,Adult
6,73587.0,1.0,2.0,38.7,94.0,2.0,202.0,21.11,Adult
7,73596.0,2.0,2.0,38.3,107.0,2.0,164.0,20.93,Adult
8,73607.0,1.0,2.0,38.9,89.0,2.0,113.0,17.47,Senior
9,73610.0,1.0,1.0,28.9,90.0,2.0,95.0,3.24,Adult


In [5]:
#lets see how age_group is divided either balanced or imbalnaced
distribution=train['age_group'].value_counts()
percentage=train['age_group'].value_counts(normalize=True) *100
for idx in distribution.index:
    print(f"class{idx}: {distribution[idx]} rows ({percentage[idx]:.2f}%)")

classAdult: 1638 rows (83.91%)
classSenior: 314 rows (16.09%)


In [6]:
# looks like highly imbalanced
#lets check the missing value now
missing_train=train.isnull().sum()
print("missing cols in train only:")
print(missing_train[missing_train>0])
print("\n")
print("missing cols in test only:")
missing_test=test.isnull().sum()
print(missing_test[missing_test>0])

missing cols in train only:
SEQN         12
RIAGENDR     18
PAQ605       13
BMXBMI       18
LBXGLU       13
DIQ010       18
LBXGLT       11
LBXIN         9
age_group    14
dtype: int64


missing cols in test only:
SEQN        2
RIAGENDR    2
PAQ605      1
BMXBMI      1
LBXGLU      1
DIQ010      1
LBXGLT      2
LBXIN       1
dtype: int64


In [7]:
#we have missing data in 14 rows of age_group which should be dropped
train_clean=train.dropna(subset=['age_group']).reset_index(drop=True)
print(f"original train rows:{train.shape[0]}")
print(f"new train rows:{train_clean.shape[0]}")

original train rows:1966
new train rows:1952


In [8]:
#lest seaperate features as X and target as Y
#drop seqn and age_group in X
feature_cols=[cols for cols in train_clean.columns if cols not in ['SEQN','age_group']]
X_train=train_clean[feature_cols]
y=train_clean['age_group']

X_test=test[feature_cols]

In [9]:
print(f"Features (X_train) shape: {X_train.shape} (This is the input data)")
print(f"Target (y) shape:   {y.shape} (This is what we want to predict)")

Features (X_train) shape: (1952, 7) (This is the input data)
Target (y) shape:   (1952,) (This is what we want to predict)


In [10]:
#lets talk about missing values now 
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy='median')
X_train_imputed_array = imputer.fit_transform(X_train)
X_train_imputed=pd.DataFrame(X_train_imputed_array,columns=X_train.columns)
print("check for x_train_imputed:\n",X_train_imputed.isnull().sum())
print("\n")
X_test_imputed_array=imputer.transform(X_test)
X_test_imputed=pd.DataFrame(X_test_imputed_array, columns=X_test.columns)
print("check for X_test_imputed:\n",X_test_imputed.isnull().sum())

check for x_train_imputed:
 RIAGENDR    0
PAQ605      0
BMXBMI      0
LBXGLU      0
DIQ010      0
LBXGLT      0
LBXIN       0
dtype: int64


check for X_test_imputed:
 RIAGENDR    0
PAQ605      0
BMXBMI      0
LBXGLU      0
DIQ010      0
LBXGLT      0
LBXIN       0
dtype: int64


In [11]:
#no need to encode here as the only two categorical cols gender and physical activity are already in 1 or 2 
#lets go for our model implemnetation
#lest start with baseline model , randomforest

In [12]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [13]:
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_imputed,y)
train_predictions=model.predict(X_train_imputed)
train_acc = accuracy_score(y, train_predictions)
print(f"Training Accuracy: {train_acc * 100:.2f}%")

Training Accuracy: 99.95%


In [14]:
#as acc is close to 100 , it shows overfitting as it capture the noise too 
#we need to split the training data in two training and validation set maintaining the class imbalnce ration by using stratified split rather than simple fit

In [15]:
from sklearn.model_selection import train_test_split
#split train and validation set in startifies split method
X_train, X_val, y_train, y_val = train_test_split(
    X_train_imputed, y, 
    test_size=0.2, 
    stratify=y, 
    random_state=42
)

#create a new randomforest model
new_model=RandomForestClassifier(n_estimators=100, random_state=42)
new_model.fit(X_train,y_train)
train_acc=accuracy_score(y_train,new_model.predict(X_train))
val_acc=accuracy_score(y_val,new_model.predict(X_val))

print(f"Training Accuracy (Studied Data):   {train_acc * 100:.2f}%")
print(f"Validation Accuracy (Unseen Data): {val_acc * 100:.2f}%")

Training Accuracy (Studied Data):   100.00%
Validation Accuracy (Unseen Data): 82.35%


In [16]:
#still the model just takes the entire noise , no improvemnts
#lets tune it by using two hyperparameters

In [17]:
#create a new model again
constrained_model=RandomForestClassifier(n_estimators=100,max_depth=5,min_samples_leaf=10,random_state=42)

constrained_model.fit(X_train,y_train)

new_train_acc=accuracy_score(y_train,constrained_model.predict(X_train))
new_val_acc=accuracy_score(y_val,constrained_model.predict(X_val))

print(f"new Training Accuracy (Studied Data):   {new_train_acc * 100:.2f}%")
print(f"new Validation Accuracy (Unseen Data): {new_val_acc * 100:.2f}%")

new Training Accuracy (Studied Data):   85.78%
new Validation Accuracy (Unseen Data): 83.89%


In [18]:
# though the validation acc jumped up and both are identical we can say model does a good work on the unseen data
#but still val_acc is 83% same as class imbalance where adults are 83% and seniors are rest 
#this shows model is justing guessing the adults and heavily inclined towards the adults like bias rather than predicting accuractly
#we need to solve this by usning class imbalance method

In [19]:
balance_model=RandomForestClassifier(n_estimators=100,max_depth=5,min_samples_leaf=10,random_state=42,class_weight='balanced')
balance_model.fit(X_train,y_train)

bal_train_acc=accuracy_score(y_train,balance_model.predict(X_train))
bal_val_acc=accuracy_score(y_val,balance_model.predict(X_val))

print(f"balance Training Accuracy (Studied Data):   {bal_train_acc * 100:.2f}%")
print(f"balance Validation Accuracy (Unseen Data): {bal_val_acc * 100:.2f}%")

balance Training Accuracy (Studied Data):   77.77%
balance Validation Accuracy (Unseen Data): 72.12%


In [20]:
#acc went down which shows no model is working to guess where seniors is present rtaher than blindly guessuing adults everywhere
#we create a confusion matrix to check how many seniors it captured

In [21]:
from sklearn.metrics import confusion_matrix, classification_report

val_preds = balance_model.predict(X_val)
cm = confusion_matrix(y_val, val_preds)
print("Confusion Matrix-----")
print(f"Predicted Adults | Predicted Seniors")
print(f"  {cm[0][0]:<14} |   {cm[0][1]}          <- Actual Adults")
print(f"  {cm[1][0]:<14} |   {cm[1][1]}          <- Actual Seniors")



Confusion Matrix-----
Predicted Adults | Predicted Seniors
  249            |   79          <- Actual Adults
  30             |   33          <- Actual Seniors


In [22]:
print(classification_report(y_val, val_preds))

              precision    recall  f1-score   support

       Adult       0.89      0.76      0.82       328
      Senior       0.29      0.52      0.38        63

    accuracy                           0.72       391
   macro avg       0.59      0.64      0.60       391
weighted avg       0.80      0.72      0.75       391



In [23]:
#now we create new feature like glucode-insulin ratio and meatbolic risj index so that we can get much better score

In [24]:
X_new = X_train_imputed.copy()
X_new['glucose_insulin_ratio']=X_new['LBXGLU']/(X_new['LBXIN']+1e-5)
X_new['bmi_glucose_index'] = X_new['BMXBMI'] * X_new['LBXGLU']

print(X_new[['glucose_insulin_ratio', 'bmi_glucose_index']].head())

   glucose_insulin_ratio  bmi_glucose_index
0               7.377594             3927.0
1              23.116823             1806.7
2              14.495090             2064.8
3               6.439624             3005.6
4               9.432226             3697.7


In [25]:
#resplit the data
X_train_eng, X_val_eng, y_train, y_val = train_test_split(
    X_new, y, 
    test_size=0.2, 
    stratify=y, 
    random_state=42
)

#reclassify the model

eng_model = RandomForestClassifier(
    n_estimators=100, max_depth=5, min_samples_leaf=10, 
    class_weight='balanced', random_state=42
)
eng_model.fit(X_train_eng, y_train)


,n_estimators,100
,criterion,'gini'
,max_depth,5
,min_samples_split,2
,min_samples_leaf,10
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [26]:
print(classification_report(y_val, eng_model.predict(X_val_eng)))

              precision    recall  f1-score   support

       Adult       0.88      0.76      0.82       328
      Senior       0.28      0.48      0.35        63

    accuracy                           0.71       391
   macro avg       0.58      0.62      0.58       391
weighted avg       0.79      0.71      0.74       391



In [27]:
#looks like f1 score went down , we added extra cols to make it easier but it gave negative output by adding noise
#now lest go to lightGBM model rather than randomforest

In [28]:
!pip install lightgbm

In [29]:
import lightgbm as lgb

lgb_model = lgb.LGBMClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.05,
    scale_pos_weight=5.2,  
    random_state=42,
    verbose=-1             
)

lgb_model.fit(X_train,y_train)
lgb_preds = lgb_model.predict(X_val)
print(classification_report(y_val, lgb_preds))

              precision    recall  f1-score   support

       Adult       0.87      0.72      0.79       328
      Senior       0.24      0.46      0.32        63

    accuracy                           0.68       391
   macro avg       0.56      0.59      0.55       391
weighted avg       0.77      0.68      0.71       391



In [30]:
#here eventhough we used powerful algo like lightgbm it gave use much lower f1score than random forest as 
#lighgbm works wee for large dataset but we have small dataset , which makes it correct and memorize even the noise too lowering f1score
#now we go back to random forst but use Stratified K-Fold Cross-Validation. rather than just startified split

In [31]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
#set up the sliptter , we use 5 fold spitting here
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

#we track score for each fold now
fold_macro_f1 = []
for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_imputed, y)):
    #split the data for train and avl in each fold
    X_tr, X_val = X_train_imputed.iloc[train_idx], X_train_imputed.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    #reintialize model:
    cv_model = RandomForestClassifier(
        n_estimators=100, 
        max_depth=5, 
        min_samples_leaf=10, 
        class_weight='balanced', 
        random_state=42
    )

    cv_model.fit(X_tr,y_tr)

    pred=cv_model.predict(X_val)

    score = f1_score(y_val, pred, average='macro')
    fold_macro_f1.append(score)
    print(f"Fold {fold + 1} Macro F1-Score: {score:.4f}")


Fold 1 Macro F1-Score: 0.5827
Fold 2 Macro F1-Score: 0.6239
Fold 3 Macro F1-Score: 0.6345
Fold 4 Macro F1-Score: 0.5868
Fold 5 Macro F1-Score: 0.6705


In [32]:
print(f"final avg f1 score: {np.mean(fold_macro_f1):.4f}")

final avg f1 score: 0.6197


In [33]:
#now lets sumbit our first model predc

final_model = RandomForestClassifier(
    n_estimators=100, 
    max_depth=5, 
    min_samples_leaf=10, 
    class_weight='balanced', 
    random_state=42
)
final_model.fit(X_train_imputed, y)

test_predictions = final_model.predict(X_test_imputed)

#create the submisiion file
submission = pd.DataFrame({'age_group': test_predictions})
submission.to_csv('submission.csv', index=False)

In [34]:
print(submission)

    age_group
0       Adult
1      Senior
2      Senior
3       Adult
4       Adult
..        ...
307     Adult
308     Adult
309     Adult
310     Adult
311    Senior

[312 rows x 1 columns]


In [35]:
#above we got text rather than 0 or 1 , so we nend to map adult to 0 and senior to 1


In [36]:
mapping = {'Adult': 0, 'Senior': 1}
submission['age_group'] = submission['age_group'].map(mapping)

In [37]:
submission

,age_group
0,0
1,1
2,1
3,0
4,0
...,...
307,0
308,0
309,0
310,0


In [38]:
submission.to_csv('submission.csv', index=False)

In [40]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier

# 1. Rebuild the final engineered datasets directly from our clean imputed arrays
X_train_final = pd.DataFrame(X_train_imputed).copy()
X_test_final = pd.DataFrame(X_test_imputed).copy()


glucose_train = X_train_final.iloc[:, 1]
insulin_train = X_train_final.iloc[:, 4]
bmi_train = X_train_final.iloc[:, 3]

glucose_test = X_test_final.iloc[:, 1]
insulin_test = X_test_final.iloc[:, 4]
bmi_test = X_test_final.iloc[:, 3]


X_train_final['Insulin_Glucose_Ratio'] = insulin_train / (glucose_train + 1e-5)
X_test_final['Insulin_Glucose_Ratio'] = insulin_test / (glucose_test + 1e-5)

X_train_final['Log_Metabolic_Product'] = np.log1p(glucose_train) * np.log1p(insulin_train)
X_test_final['Log_Metabolic_Product'] = np.log1p(glucose_test) * np.log1p(insulin_test)

X_train_final['BMI_Glucose_Stress'] = bmi_train * glucose_train
X_test_final['BMI_Glucose_Stress'] = bmi_test * glucose_test


y_numeric = pd.Series(y).map({'Adult': 0, 'Senior': 1}).to_numpy()
imbalance_ratio = np.sum(y_numeric == 0) / np.sum(y_numeric == 1)

accumulated_probs = np.zeros(len(X_test_final))
seeds = [42, 101, 2026, 777, 999]


for seed in seeds:
    seed_xgb = XGBClassifier(
        n_estimators=180,
        max_depth=4,
        learning_rate=0.04,
        scale_pos_weight=imbalance_ratio,
        random_state=seed,
        eval_metric='logloss'
    )
    seed_xgb.fit(X_train_final, y_numeric)
    accumulated_probs += seed_xgb.predict_proba(X_test_final)[:, 1]

smoothed_final_probs = accumulated_probs / len(seeds)


thresh_15 = np.percentile(smoothed_final_probs, 100 - 15)
final_preds = np.where(smoothed_final_probs >= thresh_15, 1, 0)


submission_df = pd.DataFrame({'age_group': final_preds}).astype(int)
submission_df.to_csv('submission.csv', index=False)


print(f"Total Rows: {len(submission_df)} | Total Predicted Seniors: {np.sum(final_preds == 1)}")

Total Rows: 312 | Total Predicted Seniors: 47
